# E2E 02 - Full Pipeline (Builder + Query)

Questo notebook esegue il flusso completo:
1. costruzione del Knowledge Graph con Builder Graph
2. interrogazione con Query Graph (`run_query`)
3. controlli di sanita' sulle risposte

In [ ]:
from __future__ import annotations

from pathlib import Path
import fitz

from src.graph.builder_graph import run_builder
from src.generation.query_graph import run_query


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.toml non trovato risalendo dalla cwd")


repo_root = find_repo_root(Path.cwd())
fixtures_dir = repo_root / "tests" / "fixtures"
docs_dir = fixtures_dir / "sample_docs"
ddl_file = fixtures_dir / "sample_ddl" / "simple_schema.sql"
generated_dir = repo_root / "notebooks" / "e2e-tests" / "generated"
generated_dir.mkdir(parents=True, exist_ok=True)
pdf_path = generated_dir / "business_context_full_pipeline.pdf"

In [ ]:
if not pdf_path.exists():
    text_blocks = []
    for src in [docs_dir / "business_glossary.txt", docs_dir / "data_dictionary.txt"]:
        text_blocks.append(f"# Source: {src.name}\n")
        text_blocks.append(src.read_text(encoding="utf-8"))

    doc = fitz.open()
    page = doc.new_page()
    page.insert_textbox(
        fitz.Rect(40, 40, 555, 800),
        "\n\n".join(text_blocks),
        fontsize=10,
        fontname="helv",
    )
    doc.save(pdf_path)
    doc.close()

print(f"PDF pronto: {pdf_path}")

In [ ]:
builder_state = run_builder(
    raw_documents=[str(pdf_path)],
    ddl_paths=[str(ddl_file)],
    production=False,
)

print("Tabelle completate:", builder_state.get("completed_tables", []))
assert len(builder_state.get("completed_tables", [])) > 0, "Builder non ha completato tabelle"

In [ ]:
questions = [
    "Quali entita principali sono presenti nel dominio e-commerce?",
    "Come e' collegata la tabella orders con customers?",
    "Quali colonne identificano in modo univoco un ordine?",
]

results = []
for q in questions:
    out = run_query(q)
    results.append({
        "question": q,
        "final_answer": out.get("final_answer", ""),
        "sources": out.get("sources", []),
    })

for item in results:
    print("=" * 80)
    print("Q:", item["question"])
    print("A:", item["final_answer"])
    print("Sources:", item["sources"])

In [ ]:
for item in results:
    assert item["final_answer"], f"Risposta vuota per domanda: {item['question']}"

print("Full pipeline E2E completato con successo")